# Temiz veri üzerinde MLP / LSTM / GRU / BiLSTM — 5 seed
Yalnızca bu notebook'u çalıştırın. Eski `makale.ipynb` çalıştırılmaz.

**Colab: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU.**
24 saat × 12 özellik; 70/15/15 kronolojik bölme; training üzerinde MinMaxScaler;
Adam 0.001; batch 256; shuffle=False; 150 epoch; patience 10;
seed 11,22,33,44,55. BiLSTM: önceki düzeltilmiş sürüm.

Eksik saat atlayan pencereler dışlanır. Validation/test girişleri yalnız geçmişteki
ölçülmüş saatleri kullanır; bu, bir saat ilerisi kayan tahmin değerlendirmesidir.
Yeni sonuçlar eski skorlarla doğrudan karşılaştırılmaz. Makale/FPGA işlemi yoktur.

In [ ]:
import subprocess, sys
from pathlib import Path
REPO = Path('/content/enerji-temiz-rerun')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','feat/clean-data-model-rerun','--single-branch',
                    'https://github.com/yagmurdamlakara/enerji-tuketimi-tahmini.git',str(REPO)],check=True)
print(subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True))
import torch, pandas, numpy, sklearn, joblib
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — GPU seçmeniz önerilir')

## Veri ve sonuç klasörü
Drive bağlanır; tamamlanan her seed'in çıktısı anında Drive'a yazılır.
`DATA_PATH` değerini Drive'daki ham veri dosyanıza göre değiştirin.
Dosya yoksa yükleme penceresi açılır. Her çalıştırma ayrı sonuç klasörü oluşturur.

In [ ]:
from google.colab import drive, files
from datetime import datetime, timezone
drive.mount('/content/drive')
DATA_PATH = Path('/content/drive/MyDrive/household_power_consumption.txt')
if not DATA_PATH.is_file():
    print('household_power_consumption.txt dosyasını seçin.')
    import os
    os.chdir('/content')
    uploaded = files.upload()
    DATA_PATH = Path('/content/household_power_consumption.txt')
    del uploaded
assert DATA_PATH.is_file(), f'Dosya bulunamadı: {DATA_PATH}'
OUT = Path('/content/drive/MyDrive/enerji_temiz_model_sonuclari') / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
print('Sonuçlar:', OUT)

## Eğitimi başlat — 4 model × 5 seed
Kesilirse tamamlanan seed dosyaları Drive'da kalır. Yeniden tam deney başlatmak
önceki hücreyle yeni sonuç klasörü gerektirir; otomatik kaldığı yerden devam yoktur.
Aşağıdaki hücrenin hatasız bitmesi 20 eğitimin tamamlandığını gösterir.

In [ ]:
subprocess.run([sys.executable, '-u', str(REPO/'clean_data_rerun.py'),
                '--data',str(DATA_PATH),'--output-dir',str(OUT)],check=True,cwd=REPO)

## Hocaya gönderilecek tablo ve indirilecek sonuç paketi
MAE ve RMSE kWh; R² birimsiz. ± değerleri beş seed arasında örnek standart sapmasıdır (ddof=1).
`model_test_targets.csv` sonraki baseline karşılaştırmasının tam hedef listesidir.
Bu aşamada baseline yeniden hesaplanmaz.

In [ ]:
import pandas as pd, json, shutil
assert json.loads((OUT/'run_summary.json').read_text())['status'] == 'complete'
display(pd.read_csv(OUT/'advisor_table.csv'))
display(pd.read_csv(OUT/'split_counts.csv'))
archive = shutil.make_archive('/content/'+OUT.name+'_sonuclar','zip',OUT)
files.download(archive)